# Mixed layer heat budget analysis of ACCESS-OM2 runs

This notebook contains code to analyse the mixed layer temperature budget in ACCESS-OM2 using daily diagnostics.

A short summary of the theory behind this budget and the diagnostics is contained below. More details can be found in a publication in preparation (TBC).

## Background Theory and diagnostics

### ACCESS-OM2 "Eulerian" heat budget

The budget for temperature within a single grid-cell can be formulated as:

\begin{equation}
    \frac{\partial}{\partial t}\left(\int_R \rho_0 C dV\right) = -\oint_{\partial R} \left[\rho_0 C (\mathbf{v} - \mathbf{v}^{(b)}) + \mathbf{J}\right]\cdot\mathbf{\hat{n}}\,d\mathcal{S},
\end{equation}

where $R$ represents the grid cell region, $C$ is the tracer concentration $=C_p \Theta$ for temperature, $\mathbf{\hat{n}}$ is the outward normal vector on the boundary surface $\partial R$, $d\mathcal{S}$ is the area element on that boundary, $\mathbf{v}$ is the fluid velocity, $\mathbf{v}^{(b)}$ is the velocity of the boundary and $\mathbf{J}$ represents tracer fluxes across the boundary surface associated with sub-grid scale parameterizations (such as vertical mixing) and boundary fluxes (such as air-sea tracer fluxes).

The equivalent equation in terms of MOM5 diagnostics (per unit area) is given by:
\begin{align}
  \textit{temp\_tendency} = &\textit{temp\_advection} + \\ &\quad +
  \textit{temp\_submeso} \\ &\quad + \textit{temp\_vdiffuse\_diff\_cbt}  + \textit{temp\_nonlocal\_KPP} \\ &\quad + \textit{sw\_heat} +
  \textit{temp\_rivermix} + \textit{temp\_vdiffuse\_sbc} + \textit{sfc\_hflux\_pme} \\
  & \quad + \textit{frazil\_3d}\\ 
   & \quad + \textit{temp\_vdiffuse\_k33} + \textit{neutral\_diffusion\_temp}\\
   & \quad + \textit{neutral\_gm\_temp} \\
   & \quad + \textit{mixdownslope\_temp} + \textit{temp\_sigma\_diff} + \textit{temp\_eta\_smooth}
\end{align}

All terms are in units of Wm$^{-2}$ - i.e. the tendency of the heat content within each grid cell per unit area, $\rho_0 C_p\Theta \Delta z$, where $\Delta z$ is the time variable grid cell thickness, $\rho_0=1035$kgm$^{-3}$ is the reference density, $C_p=3992.10322329649$Jkg$^{-1}$$^\circ$C$^{-1}$ is the specific heat and $\Theta$ is Conservative Temperature.

- temp\_tendency is the tendency term
- temp\_advection is the convergence of the three-dimensional resolved advection (this can be split into components by taking the convergence of the temp\_xflux\_adv, temp\_yflux\_adv and temp\_zflux\_adv terms). Note that this is equivalent to
\begin{equation}
-\oint_{\partial R - \partial\eta} \rho_0 C (\mathbf{v} - \mathbf{v}^{(b)})\cdot\mathbf{\hat{n}}\,d\mathcal{S},
\end{equation}
i.e. it does not include the dia-surface motion across the free-surface ($\partial\eta$), which is instead captured by $\textit{sfc\_hflux\_pme}$, equal to $-\oint_{\partial\eta} \rho_0 C (\mathbf{v} - \mathbf{v}^{(b)})\cdot\mathbf{\hat{n}}\,d\mathcal{S}$
- temp\_submeso is the convergence of the three-dimensional parameterized submesoscale advection (pretty small).
- temp\_vdiffuse\_diff\_cbt and temp\_nonlocal\_KPP are the vertical mixing terms.
- The next line contains all of the surface heat flux terms. Note that sw\_heat is a three-dimensional term that *redistributes* the impact of SW radiation from the surface layer into the interior (i.e. it is negative in the surface layer and positive below, summing to zero). temp\_rivermix is also three-dimensional as the impact of river runoff is spread over a few layers (4 I think). The other terms are two-dimensional (only non-zero in the surface layer).
- frazil\_3d is the formation of frazil ice
- temp\_vdiffuse\_k33 and neutral\_diffusion\_temp are parameterized along-isopycnal mixing (might not be on in all configurations, e.g. ACCESS-OM2-01).
- neutral\_gm\_temp is parameterization advection by mesoscale eddies
- The last line includes some miscellaneous mixing terms (all pretty small, and not all active depending on configuration).

Also see https://github.com/COSIMA/access-om2/issues/139#issuecomment-639278547 for a discussion of the surface heat flux terms in ACCESS-OM2/CM2. There are some cells below which check this budget closure.

### Mixed layer temperature budget 

The mixed layer depth is defined as the depth at which the buoyancy difference to from the surface is $0.0003$ms$^{-2}$, corresponding to a density difference of $0.031$kgm$^{-3}$. 
$0.03$kgm$^{-3}$ is a widely used value in the literature. We define the mixed layer in a continuous sense, such that the mixed layer base can lie between two grid cells (linear interpolation), with a known fractional contribution of the bottom grid cell to the mixed layer. The mixed layer within a given model column is defined by the region $\partial R_H$.

We are most interested in the tracer concentration averaged over the mixed layer volume, rather than the total mixed layer tracer content.
We define this (for temperature) as,
\begin{equation}
    \Theta_H \equiv \frac{1}{H}\int_{-H_z}^\eta \Theta dz,
\end{equation}
where $\eta$ is the free-surface height and $H_z$ is the depth of the mixed layer (from $z=0$), such that the total mixed layer depth is $H=\eta+H_z$.

A bunch of maths, shown elsewhere (paper in preparation), shows that $\Theta_H$ obeys the budget equation,

\begin{align}
        \frac{\partial \Theta_H}{\partial t}&= \quad\quad &\quad\quad\text{tendency}\\ &\quad-\frac{1}{AH}\oint_{\partial R_H - \partial\eta} \left(\Theta-\Theta_H\right) \mathbf{v}\cdot\mathbf{\hat{n}}\,d\mathcal{S}-\frac{Q_{\text{L}}}{\rho_0 H}\quad\quad &\quad\quad\text{advection (+ eddy processes)} \\
        &\quad+\frac{Q_\text{net}}{\rho_0 H}-\frac{\Theta_a - \Theta_H}{\rho_0H}Q_m\quad\quad &\quad\quad\text{surface fluxes} \\
        &\quad-\frac{Q_{\text{SWP}}}{\rho_0 H}\quad\quad &\quad\quad\text{shortwave penetration} \\
        &\quad-\frac{Q_\text{mix}}{\rho_0 H}\quad\quad &\quad\quad\text{vertical mixing} \\
        &\quad-\frac{\Theta_{\text{ent}}-\Theta_H}{H}\frac{\partial H_z}{\partial t}\quad\quad &\quad\quad\text{entrainment}
\end{align}
where $A$ is the area of the grid cell and other terms are described below, in order.

#### Term 1: 
The first line is the tendency term. Note that this term is not a diagnostic in MOM5 (it is not temp\_tendency, which is the tendency of the total heat content of the grid cells making up the mixed layer, $R_{\Sigma G}$ in the paper) and needs to be computed offline (from snapshots or time-averages, depending on whether standard or hat-averaging is used on the budget diagnostics) using the mixed layer temperature diagnostic.

#### Term 2: 
The second line represents advection and eddy driven processes. This term is effectively equal to $\textit{temp\_advection} + \textit{temp\_submeso} + \textit{temp\_vdiffuse\_k33} + \textit{neutral\_diffusion\_temp} + \textit{neutral\_gm\_temp}$ divided by $C_p\rho_0 H$, except that since $H$ is time-varying, this division needs to be done daily at every time-step. This is done by the new diagnostics $\textit{temp\_advection\_in\_mld}$ (and equivalent for the eddy terms), which are equal to $\textit{temp\_advection}/H$ summed over the mixed layer (so divide these by $\rho_0 C_p$ to get the above equation). Hwoever, we also note that more maths (see the appendix of the paper), shows that two additional correction terms,
\begin{equation}
\frac{\Theta_H}{H}\nabla\cdot\mathbf{U} - \frac{\Theta_{\text{ent}}}{H} w^{(s)}_H \quad\quad\quad\text{(TO COMPUTE!!!)}
\end{equation}
need to be added to $\textit{temp\_advection\_in\_mld}/\rho_0/C_p$ in order to get the advection term in the above equation to account for the fact that the advection diagnostic temp\_advection is computed in GVC coordinates, not Eulerian coordinates. $\nabla\cdot\mathbf{U}$ represents the divergence of the vertically-integrated transport (throughout the whole ocean depth) and $w^{(s)}_H$ represents the motion of the GVC coordinates at the base of the mixed layer (easily computable from $\partial\eta/\partial t$). $\Theta_{\text{ent}}$ is somewhat tricky to define/compute, so we will probably ignore this last term as it should be small. Also worth noting that both of these terms are dependent on the temperature scale, and thus remove the dependence of the temperature scale in $\textit{temp\_advection\_in\_mld}$.

#### Term 3: 
The third line represents surface fluxes, including the terms $\textit{temp\_vdiffuse\_sbc\_in\_mld} + \textit{sfc\_hflux\_pme\_in\_mld} + \textit{frazil\_3d\_in\_mld} + \textit{temp\_rivermix\_in\_mld}$ divided by $\rho_0 C_p$. The second part of this term in the above equation accounts for the impact of surface mass fluxes, $Q_m$ (in kgs$^{-1}$), where $C_a$ is the tracer concentration of the added (or removed) surface mass. For temperature in MOM5, $C_a$ is equal to the temperature of the top grid cell. In this term, another correction needs to be added to the existing daily diagnostics, 
\begin{equation}
\Theta_H Q_m/(\rho_0 H).\quad\quad\quad\text{(TO COMPUTE!!!)}
\end{equation} 
Again, due to the fact that $Q_m$, $\Theta_H$ and $H$ all vary in time, this term needs to be computed daily. Note that for simplicity (and because its tiny) we include $\textit{temp\_eta\_smooth\_in\_mld}$ in this term as well.

#### Term 4: 
The fourth line represents shortwave penetration, simply equal to $\textit{sw\_heat\_in\_mld}/\rho_0/C_p$.

#### Term 5: 
The fifth line represents vertical mixing at the base of the mixed layer, equal to $(\textit{temp\_vdiffuse\_diff\_cbt\_in\_mld}  + \textit{temp\_nonlocal\_KPP\_in\_mld})/\rho_0/C_p$.

#### Term 6: 
The last line represents entrainment, where $\Theta_\text{ent}$ is the temperature of the entrained water. This is not an explicit diagnostic in MOM5 (computing $\Theta_{\text{ent}}$ would be tricky). So here, we compute it as the residual of the budget above.

Note that the term $\textit{temp\_tendency\_in\_mld}$ is the sum of all the \_in\_mld terms, which represents (apart from the 3 corrections above) the tendency in the heat content of the layer ignoring the extra heat content entering through entrainment. Hence (again ignoring the 3 corrections noted above) entrainment effectively represents the difference between $\textit{temp\_tendency\_in\_mld}$ and $\partial \Theta_H/\partial t$.

#### On the corrections
None have been computed as yet (Jan 2025). The first part of the advection one, and the P-E+R one, just correspond to terms in the free-surface evolution equation
\begin{equation}
\frac{\partial \eta}{\partial t} = \nabla\cdot\mathbf{U} + Q_m/\rho_0
\end{equation}
multiplied by $\Theta_H/H$. The term dependent on $w^{(s)}_H$ is hard to compute (depends on $\Theta_{\text{ent}}$) and so we'll probably ignore it...

## Hat averaging

The following material comes from Bladwell et al. (2025, in prep.). Consider a variable $\xi(t)$ (e.g. mixed layer temperature at a single location), defined as a function of time. We will consider two ``epochs" defined by the time periods $t\in(t_1,t_1+\Delta t_1)$ and  $t\in(t_2,t_2+\Delta t_2)$. The standard average of the tendency of $\xi$ between $t_1$ and $t_2+\Delta t_2$ (i.e. over the entire period covered by standard tendency diagnostics), multiplied by the time gap between them, is given by:
\begin{equation}
(t_2+\Delta t_2 - t_1)\overline{\frac{\partial\xi}{\partial t}}^{t_1,t_2+\Delta t_2} \equiv \int_{t_1}^{t_2+\Delta t_2} \frac{\partial\xi}{\partial t} dt = \left[\xi(t_2+\Delta t_2) - \xi(t_1)\right]
\end{equation}
This corresponds to a difference in snapshots of $\xi$, and thus is not typically a quantity of interest.

Instead, we define the "hat average" operator between the two epochs as,
\begin{equation}
\hat{\frac{\partial\xi}{\partial t}}^{t_1,t_1+\Delta t_1,t_2,t_2+\Delta t_2} \equiv \int_{t_1}^{t_1+\Delta t_1} \frac{t-t_1}{\Delta t_1}\frac{\partial\xi}{\partial t} dt + \int_{t_1+\Delta t_1}^{t_2} \frac{\partial\xi}{\partial t} dt + \int_{t_2}^{t_2+\Delta t_2} \frac{t_2+\Delta t_2 - t}{\Delta t_2}\frac{\partial\xi}{\partial t} dt = \overline{\xi}^{t_2,t_2+\Delta t_2} - \overline{\xi}^{t_1,t_1+\Delta t_1}
\end{equation}
This corresponds to a "rising average" over the first epoch, and standard average between them, and a falling average over the second epoch (hence the "hat average"). Evidently, the hat average between the two epochs is the operation needed to relate the tendency $\partial\xi/\partial t$ to the difference between the values of $\xi$ averaged over the two epochs.

Note that the diagnostics output from MOM5 to form the hat averaging correspond to standard, rising and falling averages over a shorter, pre-defined time period (below daily) that does not usually correspond to the epoch differences of interest (e.g. differences between months). If our longer epoch of interest, say $t\in(t_1,t_1+\Delta t_1)$, consists of $N$ sections of shorter diagnostics of length $\Delta t$ (e.g. the month of January consists of $N=31$ sections of length $\Delta t=1$ day, with $\Delta t_1=N\Delta t$), then we can use the following formula's to compute the rising "difference" (i.e. the first term in the previous equation) over the longer (i.e. entire January) period from the rising averages over each day,
\begin{equation}
\int_{t_1}^{t_1+N\Delta t} \frac{t-t_1}{N\Delta t} \frac{\partial\xi}{\partial t}dt = \sum_{n=1}^N \frac{1}{N} \int_{t_1+(n-1)\Delta t}^{t_1+n\Delta t} \frac{t-(t_1+(n-1)\Delta t)}{\Delta t} \frac{\partial\xi}{\partial t} dt + \sum_{n=1}^N \frac{(n-1)}{N} \int_{t_1+(n-1)\Delta t}^{t_1+n\Delta t} \frac{\partial\xi}{\partial t} dt
\end{equation}
where the LHS represents the long rising difference over the period $t\in(t_1,t_1+\Delta t_1)$, the first term on the RHS is the sum of $1/N$ times the short rising difference over the short period plus $(n-1)/N$ times the short standard difference over the short period.
The long standard difference is trivially,
\begin{equation}
\int_{t_1}^{t_1+N\Delta t} \frac{\partial\xi}{\partial t}dt = \sum_{n=1}^N \int_{t_1+(n-1)\Delta t}^{t_1+n\Delta t} \frac{\partial\xi}{\partial t} dt
\end{equation}
the long falling difference can be computed as the difference between the two previous equations
\begin{equation}
\int_{t_1}^{t_1+N\Delta t} \frac{t_1 + N\Delta t - t}{N\Delta t} \frac{\partial\xi}{\partial t}dt = \int_{t_1}^{t_1+N\Delta t} \frac{\partial\xi}{\partial t}dt - \int_{t_1}^{t_1+N\Delta t} \frac{t-t_1}{N\Delta t} \frac{\partial\xi}{\partial t}dt
\end{equation}


In [1]:
#Load required packages
%matplotlib inline
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import pandas as pd
import cftime
from tqdm import tqdm

import cmocean as cm
import sys, os
import datetime

from dask.distributed import Client

In [2]:
# Load workers:
client = Client(n_workers=14)
client

/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/lib/python3.10/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 37513 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/37513/status,
Dashboard: /proxy/37513/status,Workers: 14
Total threads: 14,Total memory: 63.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:41153,Workers: 14
Dashboard: /proxy/37513/status,Total threads: 14
Started: Just now,Total memory: 63.00 GiB
Comm: tcp://127.0.0.1:45493,Total threads: 1
Dashboard: /proxy/39237/status,Memory: 4.50 GiB
Nanny: tcp://127.0.0.1:40631,


2025-08-15 11:06:07,971 - distributed.protocol.core - CRITICAL - Failed to Serialize
Traceback (most recent call last):
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/lib/python3.10/site-packages/distributed/protocol/core.py", line 109, in dumps
    frames[0] = msgpack.dumps(msg, default=_encode_default, use_bin_type=True)
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/lib/python3.10/site-packages/msgpack/__init__.py", line 36, in packb
    return Packer(**kwargs).pack(o)
  File "msgpack/_packer.pyx", line 294, in msgpack._cmsgpack.Packer.pack
  File "msgpack/_packer.pyx", line 300, in msgpack._cmsgpack.Packer.pack
  File "msgpack/_packer.pyx", line 297, in msgpack._cmsgpack.Packer.pack
  File "msgpack/_packer.pyx", line 264, in msgpack._cmsgpack.Packer._pack
  File "msgpack/_packer.pyx", line 231, in msgpack._cmsgpack.Packer._pack
  File "msgpack/_packer.pyx", line 264, in msgpack._cmsgpack.Packer._pack
  File "msgpack/_packer.pyx", line 272, in msgpa

In [5]:
# Chdir for figure saving:
#os.chdir('access-om2-analysis/access-om2-sst-budget')

# Load data

### Define paths, region to analyse and time period to analyse

In [6]:
base = '/g/data/e14/rmh561/access-om2/archive/025deg_jra55_iaf_cycle6_online_mlt/'
output = 370 # 370 = 2023 - contains 3D budget diagnostics for quantifying correlation errors

tmp_folder = '/g/data/e14/rmh561/mlt_budget_temporary/'

base2 = base + 'output%03d/ocean/' % output
clim_base = base + 'clim_1989-2018/'

# Subsample regions:
#reg = [-100, 20, 0, 60] # North Atlantic
#reg = [-100, -40, 0, 30] # North Atlantic
#reg = [-230, -190, -50, -10] # EAC
#reg = [135-360,175-360, -60, -20] # SE Aus (Kajtar et al. 2022)
reg = [None,None,None,None] # Globe

# Subsample time:
#times = slice('2017-09-01',None)
#times_snap = slice('2017-09-01',None) # Note; this must be 1 more than times.
#times = slice('2023-04-01','2023-04-30')
#times_snap = slice('2023-04-01','2023-05-01') # Note; this must be 1 more than times.
times = slice(None,None)
times_snap = slice(None,None) # Note; this must be 1 more than times.

chunks2D = {'time':1,'yt_ocean':216,'xt_ocean':240}
chunks3D = {'time':1,'st_ocean':25,'yt_ocean':324,'xt_ocean':360}

### Load grid and set constants

In [7]:
ds_grid = xr.open_dataset(base2 + 'ocean_grid.nc',chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))#.isel(time=times)
rho0 = 1035.
Cp = 3992.10322329649

### Load daily data

In [8]:
# Standard average daily budget diagnostics:
ds_day_budget = xr.open_dataset(base2 + 'ocean_budget_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

# Falling average daily budget daignostics (while the name of the averaging is "risavg", in effect this is actually the falling average diagnostics):
#ds_day_budget_falavg = xr.open_dataset(base2 + 'ocean_budget_daily_risavg.nc',decode_times=False).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

# Standard daily diagnostics:
ds_day = xr.open_dataset(base2 + 'ocean_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

# Snapshots for standard average tendency computation:
ds_day_snapshot = xr.open_dataset(base2 + 'ocean_snapshot_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
# Add previous output for last element:
ds_day_snapshot_m1 = xr.open_dataset(base2.replace(str(output),str(output-1)) + 'ocean_snapshot_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_day_snapshot = xr.concat([ds_day_snapshot_m1.isel(time=-1),ds_day_snapshot],dim='time')

# Fix time variable by decoding time by hand (see https://forum.access-hive.org.au/t/cftime-vs-datetime64-time-encoding-issues-with-access-om2-025-omip-2-run/4085);
ds_day_budget = ds_day_budget.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day_budget.time.values]})
#ds_day_budget_falavg = ds_day_budget_falavg.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day_budget_falavg.time.values]})
ds_day = ds_day.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day.time.values]})
ds_day_snapshot = ds_day_snapshot.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day_snapshot.time.values]})

# Fix average_DT by decoding by hand:
ds_day_budget.average_DT.data = ds_day_budget.average_DT*np.timedelta64(1,'D')
#ds_day_budget_falavg.average_DT.data = ds_day_budget_falavg.average_DT*np.timedelta64(1,'D')
ds_day.average_DT.data = ds_day.average_DT*np.timedelta64(1,'D')

# Subselect time period:
ds_day_budget = ds_day_budget.sel(time=times)
#ds_day_budget_falavg = ds_day_budget_falavg.sel(time=times)
ds_day = ds_day.sel(time=times)
ds_day_snapshot = ds_day_snapshot.sel(time=times_snap)

/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/lib/python3.10/site-packages/xarray/core/dataset.py:274: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 1. This could degrade performance. Instead, consider rechunking after loading.
  warnings.warn(
/jobfs/147088131.gadi-pbs/ipykernel_2384019/3627627326.py:17: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  ds_day_budget = ds_day_budget.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day_budget.time.values]})
/jobfs/147088131.

### Compute climatologies (xarray, including budgets):

In [9]:
base = '/g/data/e14/rmh561/access-om2/archive/025deg_jra55_iaf_cycle6_daily_mlt/'
outputs = np.arange(336,366)
dest_folder = base + 'clim_1989-2018/'

fields = {#'ocean_budget_month.nc':'all',
          'ocean_budget_month_risavg.nc':'all',
          'ocean_snapshot_month.nc':'all',
          'ocean_month.nc':['temp_in_mld','salt_in_mld','mld']
         }

In [10]:
for file in fields.keys():
    print('Doing ' + file + '...')
    ds = xr.open_dataset(base + 'output%03d' % outputs[0] + '/ocean/' + file,decode_times=False)
    if fields[file] != 'all':
        ds = ds[fields[file]]
    ds.load()

    for output in tqdm(outputs[1:]):
        ds2 = xr.open_dataset(base + 'output%03d' % output + '/ocean/' + file,decode_times=False)
        if fields[file] != 'all':
            ds2 = ds2[fields[file]]
        ds2.load()
        ds2 = ds2.assign_coords({'time':ds.time})
        ds = ds + ds2

    ds = ds/len(outputs)
    ds.to_netcdf(dest_folder + file)        

Doing ocean_budget_month_risavg.nc...


FileNotFoundError: [Errno 2] No such file or directory: '/g/data/e14/rmh561/access-om2/archive/025deg_jra55_iaf_cycle6_daily_mlt/output336/ocean/ocean_budget_month_risavg.nc'

### Load climatologies:

In [ ]:
# Standard average daily budget diagnostics:
ds_clim_budget = xr.open_dataset(clim_base + 'ocean_budget_month.nc',decode_times=False).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_clim_budget_falavg = xr.open_dataset(clim_base + 'ocean_budget_month_risavg.nc',decode_times=False).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_clim = xr.open_dataset(clim_base + 'ocean_month.nc',decode_times=False).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_clim_snapshot = xr.open_dataset(clim_base + 'ocean_snapshot_month.nc',decode_times=False).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).load()

# Fix time variable by decoding time by hand (see https://forum.access-hive.org.au/t/cftime-vs-datetime64-time-encoding-issues-with-access-om2-025-omip-2-run/4085);
ds_clim_budget = ds_clim_budget.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_clim_budget.time.values]})
ds_clim_budget_falavg = ds_clim_budget_falavg.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_clim_budget_falavg.time.values]})
ds_clim = ds_clim.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_clim.time.values]})
ds_clim_snapshot = ds_clim_snapshot.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_clim_snapshot.time.values]})

# Add wrap-around for clim_snapshot:
tminus1 = str(ds_clim_snapshot.time.isel(time=-1).values)
tminus1 = np.datetime64(str(int(tminus1[:4])-1) + tminus1[4:])
ds_clim_snapshot = xr.concat([ds_clim_snapshot.isel(time=-1).assign_coords({'time':tminus1}),ds_clim_snapshot],dim='time')

# Decode average_DT:
ds_clim_budget.average_DT.data = ds_clim_budget.average_DT*np.timedelta64(1,'D')
ds_clim_budget_falavg.average_DT.data = ds_clim_budget_falavg.average_DT*np.timedelta64(1,'D')

### Compute climatology for a few variables (old - run in terminal)

In [ ]:
# Make a simple climatology:
# cd /g/data/e14/rmh561/access-om2/archive/025deg_jra55_iaf_cycle6_daily_mlt/
# mkdir clim_1980-2009

#fnames = ['output%03d/ocean/ocean_month.nc' % x for x in np.arange(327,357)] # 1980-2009
fnames = ['output%03d/ocean/ocean_budget_month.nc' % x for x in np.arange(336,365)] # 1989-2018
' '.join(fnames)

# ncea -v temp_in_mld output327/ocean/ocean_month.nc output328/ocean/ocean_month.nc output329/ocean/ocean_month.nc output330/ocean/ocean_month.nc output331/ocean/ocean_month.nc output332/ocean/ocean_month.nc output333/ocean/ocean_month.nc output334/ocean/ocean_month.nc output335/ocean/ocean_month.nc output336/ocean/ocean_month.nc output337/ocean/ocean_month.nc output338/ocean/ocean_month.nc output339/ocean/ocean_month.nc output340/ocean/ocean_month.nc output341/ocean/ocean_month.nc output342/ocean/ocean_month.nc output343/ocean/ocean_month.nc output344/ocean/ocean_month.nc output345/ocean/ocean_month.nc output346/ocean/ocean_month.nc output347/ocean/ocean_month.nc output348/ocean/ocean_month.nc output349/ocean/ocean_month.nc output350/ocean/ocean_month.nc output351/ocean/ocean_month.nc output352/ocean/ocean_month.nc output353/ocean/ocean_month.nc output354/ocean/ocean_month.nc output355/ocean/ocean_month.nc output356/ocean/ocean_month.nc clim_1980-2009/ocean_month.temp_in_mld.ncea.nc

In [ ]:
# Load climatology of mixed layer temperature:
ds_clim = xr.open_dataset(base + 'clim_1980-2009/ocean_month.temp_in_mld.mld.ncea.nc',decode_times=False).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_clim = ds_clim.assign_coords({'time':np.arange(1,13)})

### Load 3D budget data (only used for error checking)

In [11]:
# # monthly data:
ds_mon_budget_3d = xr.open_dataset(base2 + 'ocean_budget_month_3d.nc',decode_times=False,chunks=chunks3D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_day_budget_3d = xr.open_dataset(base2 + 'ocean_budget_daily_3d.nc',decode_times=False,chunks=chunks3D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_mon = xr.open_dataset(base2 + 'ocean_month.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

# Snapshots for standard average tendency computation:
ds_mon_snapshot = xr.open_dataset(base2 + 'ocean_snapshot_month.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
# Add previous output for last element:
ds_mon_snapshot_m1 = xr.open_dataset(base2.replace(str(output),str(output-1)) + 'ocean_snapshot_month.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_mon_snapshot = xr.concat([ds_mon_snapshot_m1.isel(time=-1),ds_mon_snapshot],dim='time')

# Fix time variable by decoding time by hand (see https://forum.access-hive.org.au/t/cftime-vs-datetime64-time-encoding-issues-with-access-om2-025-omip-2-run/4085);
ds_mon_budget_3d = ds_mon_budget_3d.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_mon_budget_3d.time.values]})
ds_day_budget_3d = ds_day_budget_3d.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day_budget_3d.time.values]})
ds_mon = ds_mon.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_mon.time.values]})
ds_mon_snapshot = ds_mon_snapshot.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_mon_snapshot.time.values]})

# Fix average_DT by decoding by hand:
ds_mon_budget_3d.average_DT.data = ds_mon_budget_3d.average_DT*np.timedelta64(1,'D')
ds_day_budget_3d.average_DT.data = ds_day_budget_3d.average_DT*np.timedelta64(1,'D')
ds_mon.average_DT.data = ds_mon.average_DT*np.timedelta64(1,'D')

# Subselect time period:
ds_mon_budget_3d = ds_mon_budget_3d.sel(time=times)
ds_day_budget_3d = ds_day_budget_3d.sel(time=times)
ds_mon = ds_mon.sel(time=times)
ds_mon_snapshot = ds_mon_snapshot.sel(time=times_snap)

/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/lib/python3.10/site-packages/xarray/core/dataset.py:274: UserWarning: The specified chunks separate the stored chunks along dimension "yt_ocean" starting at index 324. This could degrade performance. Instead, consider rechunking after loading.
  warnings.warn(
/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/lib/python3.10/site-packages/xarray/core/dataset.py:274: UserWarning: The specified chunks separate the stored chunks along dimension "xt_ocean" starting at index 360. This could degrade performance. Instead, consider rechunking after loading.
  warnings.warn(
/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/lib/python3.10/site-packages/xarray/core/dataset.py:274: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 1. This could degrade performance. Instead, consider rechunking after loading.
  warnings.warn(
/jobfs/147088131.gadi-pbs/ipykernel_2384019/3068461509.

# Construct daily, ML-binned, budgets

Here we load in all the required ML budget terms, for various time periods

## Define budget term groupings
This cell defines how the raw MOM5 budget terms (10+) are grouped into smaller groups (advection, mixing, surface forcing etc.)

In [12]:
bud_var_grps = {'advection':['temp_advection_in_mld',
                             'temp_submeso_in_mld',
                             'neutral_diffusion_in_mld_temp',
                             'neutral_gm_in_mld_temp',
                             'temp_vdiffuse_k33_in_mld'],
                'vert_mixing':['temp_nonlocal_KPP_in_mld',
                               'temp_vdiffuse_diff_cbt_in_mld'],
                'surface_flux':['temp_rivermix_in_mld',
                                'temp_vdiffuse_sbc_in_mld', 
                                'frazil_3d_in_mld',
                                'sfc_hflux_pme_in_mld',
                                'temp_eta_smooth_in_mld'], 
                'sw_pen':['sw_heat_in_mld']}

## Define functions to compute MLT budget

In [13]:
def mlt_budget_fixedh(ds_day_budget):
    """
    Compute fixedh budget terms, including residual. Does not include entrainment or mlt tendency.
    """
    # Extract variable sums in groups, dividing by rho0*Cp to convert to degC/second
    mlt_budget = (ds_day_budget['temp_tendency_in_mld']/rho0/Cp).rename('fixedh_tendency').to_dataset()
    for var in bud_var_grps.keys():
        mlt_budget[var] = ds_day_budget[bud_var_grps[var][0]]/rho0/Cp
        if (len(bud_var_grps[var])>1):
            for raw_var in bud_var_grps[var][1:]:
                mlt_budget[var] += ds_day_budget[raw_var]/rho0/Cp
    
    # Compute residual for check:
    mlt_budget['residual'] = mlt_budget['fixedh_tendency'].copy(deep=True)
    for var in list(mlt_budget.data_vars):
        mlt_budget['residual'] -= mlt_budget[var]

    return(mlt_budget)

def compute_tendency_entrainment(mlt_budget,mlt_snap):
    """
    Compute tendency term from snapshots and entrainment by residual.
    """

    mlt_snap = mlt_snap.transpose(*mlt_budget['fixedh_tendency'].dims)

    # Compute mlt tendency by taking time derivative of snapshot mlt:
    mlt_budget['mlt_tendency'] = xr.zeros_like(mlt_budget.fixedh_tendency).copy(deep=True)
    mlt_budget['mlt_tendency'].data = mlt_snap.isel(time=slice(1,None)).values - mlt_snap.isel(time=slice(0,-1)).values
    # mlt_budget['mlt_tendency'] = mlt_budget['mlt_tendency']/(ds_day_budget.average_DT/np.timedelta64(1,'s')) # Note: Depending on time decoding this line may change
    DT = xr.zeros_like(mlt_budget['mlt_tendency'].time)
    DT.data = (mlt_snap.time.isel(time=slice(1,None)).values-mlt_snap.time.isel(time=slice(0,-1)).values)/np.timedelta64(1,'s')
    mlt_budget['mlt_tendency'] = mlt_budget['mlt_tendency']/DT

    # Compute entrainment by residual:
    mlt_budget['entrainment'] = -(mlt_budget['fixedh_tendency'] - mlt_budget['mlt_tendency'])

    return(mlt_budget)

## Compute daily budget, standard averaging

The following cell loads/computes the budget terms from daily data using standard averaging and the functions above. The resulting dataset `mlt_budget_stavg_daily` will contain x*y*t arrays with the above budget groups (e.g. advection, surface forcing etc.) as well as:
- fixedh\_tendency: The tendency term of mixed layer temperature with a fixed ML depth (i.e. the sum of all the RHS terms listed above).
- residual: The residual, fixedh\_tendency minus all the RHS terms. This should be exactly zero. If it isn't, you're missing terms in the source MOM5 budget or something has gone wrong.
- mlt\_tendency: The tendency of the mixed layer temperature computed (in this case) from snapshots of the mixed layer temperature (the diagnostic temp\_in\_mld) at the start and end of the day.
- entrainment: The entrainment term, computed by residual mlt\_tendency - fixedh\_tendency (with the later here equal to the sum of the RHS terms, so the mlt budget closes)

All terms have units of degC/second.

In [14]:
# Compute in one go:
mlt_budget_stavg_daily = mlt_budget_fixedh(ds_day_budget)
mlt_budget_stavg_daily = compute_tendency_entrainment(mlt_budget_stavg_daily,ds_day_snapshot.temp_in_mld/rho0)
mlt_budget_stavg_daily.load();

/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/lib/python3.10/site-packages/distributed/client.py:3357: UserWarning: Sending large graph of size 4.24 GiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


FutureCancelledError: ('truediv-0f6bbf2e7b48f30691ae0fabf857da8d', 34, 1, 2) cancelled for reason: scheduler-connection-lost.
Client lost the connection to the scheduler. Please check your connection and re-run your work.

In [ ]:
# Compute in blocks (e.g. if doing a whole year):

# NOTE: THIS still seems way slower than it should be... Something simple might make it faster...
bs = 30; tl = len(ds_day_budget.time)
blocks = [range(tl)[x*bs:(x+1)*bs] for x in range(int(np.ceil(tl/bs)))]
blocks_snap = [range(tl+1)[x*bs:(x+1)*bs + 1] for x in range(int(np.ceil(tl/bs)))]

mlt_budget_stavg_daily_uncat = []
for i in tqdm(range(len(blocks))):
    bud = mlt_budget_fixedh(ds_day_budget.isel(time=blocks[i]))
    bud = compute_tendency_entrainment(bud,ds_day_snapshot.temp_in_mld.isel(time=blocks_snap[i])/rho0)
    mlt_budget_stavg_daily_uncat.append(bud.load())
mlt_budget_stavg_daily = xr.concat(mlt_budget_stavg_daily_uncat,dim='time')

In [ ]:
# Save to file if desired:
#mlt_budget_stavg_daily.to_netcdf(tmp_folder + 'mlt_budget_stavg_daily_online_2023.nc')

## Compute monthly budget climatology

In [ ]:
# Compute fixedh budget:
mlt_budget_stavg_clim = mlt_budget_fixedh(ds_clim_budget)

# Compute entrainment and tendnecy:
mlt_budget_stavg_clim = compute_tendency_entrainment(mlt_budget_stavg_clim,ds_clim_snapshot.temp_in_mld/rho0)

# Force computation:
mlt_budget_stavg_clim.load();

## Compute monthly difference budget, standard averaging
`mlt_budget_stavg_monthly` is simply the time integral of of the daily `mlt_budget_stavg_daily` budget. The resulting array contains the temperature difference induced by each term (or the temperature difference itself, for mlt\_tendency) across the month. Units are degC.

In [ ]:
mlt_budget_stavg_monthly = (mlt_budget_stavg_daily*(ds_day_budget.average_DT/np.timedelta64(1,'s'))).resample(time='1ME').sum()
mlt_budget_stavg_monthly['time'] = mlt_budget_stavg_daily.time.resample(time='1ME').mean() # Centre time in the middle of the month.

## Compute daily budget, hat averaging

Compute the hat-averaged daily budget. This budget (containing the same fields as `mlt_budget_stavg_daily`) corresponds to the tendency of the *daily-averaged* mixed layer temperature. E.g., mlt\_tendency in `mlt_budget_hatavg_daily` is the difference in daily-averaged temperature between the day after and the day before the time stamp (with the time stamp being at 0Z inbetween the two days), divided by the number of seconds in the day (units degC/second). The other terms are the budget contributions to this tendency. 

In [ ]:
def hat_average(st_avg,fal_avg_raw,average_DT):
    """
    Compute hat average from standard and falling average for a particular field
    """
    fal_avg = fal_avg_raw/(average_DT/np.timedelta64(1,'s')) # This line fixes a bug in the normalization of the daily falling average diagnostics
                                                             # take mean by dividing by averaging period. We do this here to avoid loading all the variables just to do this fix.
    ris_avg = st_avg - fal_avg                               # Rising = standard - falling
    hat_avg = ris_avg.isel(time=slice(0,-1)).values +  fal_avg.isel(time=slice(1,None)) # Hat = rising over first day - falling over second day              
                                                                                        # Note: Dealing with time is done outside this function
    return(hat_avg)

In [ ]:
# Template variable (note that hat average tendencies lie on snapshot (1:end-1) time:
mlt_budget_hatavg_daily = np.nan*xr.zeros_like(ds_day_snapshot.temp_in_mld.isel(time=slice(1,-1)).transpose(*ds_day_budget['temp_tendency_in_mld'].dims)) 

# Compute hat average fixedh tendency:
mlt_budget_hatavg_daily.data = hat_average(ds_day_budget['temp_tendency_in_mld'],ds_day_budget_falavg['temp_tendency_in_mld'],ds_day_budget_falavg.average_DT)/rho0/Cp

# Make a dataset:
mlt_budget_hatavg_daily = mlt_budget_hatavg_daily.rename('fixedh_tendency').to_dataset()

# Do other variables:
for var in bud_var_grps.keys():
    mlt_budget_hatavg_daily[var] = xr.zeros_like(mlt_budget_hatavg_daily['fixedh_tendency']).copy(deep=True)
    mlt_budget_hatavg_daily[var].data = hat_average(ds_day_budget[bud_var_grps[var][0]],ds_day_budget_falavg[bud_var_grps[var][0]],ds_day_budget_falavg.average_DT)/rho0/Cp
    if (len(bud_var_grps[var])>1):
        for raw_var in bud_var_grps[var][1:]:
            mlt_budget_hatavg_daily[var].data += hat_average(ds_day_budget[raw_var],ds_day_budget_falavg[raw_var],ds_day_budget_falavg.average_DT)/rho0/Cp

# Compute residual for check:
mlt_budget_hatavg_daily['residual'] = mlt_budget_hatavg_daily['fixedh_tendency'].copy(deep=True)
for var in list(mlt_budget_hatavg_daily.data_vars):
    mlt_budget_hatavg_daily['residual'] -= mlt_budget_hatavg_daily[var]

# Compute mlt tendency:
mlt = (ds_day.temp_in_mld/rho0).transpose(*mlt_budget_hatavg_daily['fixedh_tendency'].dims)
mlt_budget_hatavg_daily['mlt_tendency'] = xr.zeros_like(mlt_budget_hatavg_daily['fixedh_tendency']).copy(deep=True)
mlt_budget_hatavg_daily['mlt_tendency'].data = mlt.isel(time=slice(1,None)).values - mlt.isel(time=slice(0,-1)).values
mlt_budget_hatavg_daily['mlt_tendency'] = mlt_budget_hatavg_daily['mlt_tendency']/86400. # Note: this will only work for daily averaging, 
                                                                                           # as it assumes a 86400 time difference between 
                                                                                           # the centre of the two time-averaged periods (day before to day after)

# Entrainment term by residual:
mlt_budget_hatavg_daily['entrainment'] = -(mlt_budget_hatavg_daily['fixedh_tendency'] - mlt_budget_hatavg_daily['mlt_tendency'])

## Compute monthly difference budget, hat averaging

This section computes the monthly difference budgets (e.g. as above, contributions to the temperature differences across individual months) for standard averaging and hat averaging from the daily budgets. It then defines a function `monthly\_hat\_average` that takes these budgets as inputs, along with the monthly-averaged mixed layer temperature and two month indexes of interest, and outputs the contributions to the budget that govern the difference between the monthly-averaged mixed layer temperature of those two months.

Note: Some code and calculations here are repeated from the daily averaging performed above for conveninence.

In [ ]:
# Daily standard average budget (duplicated from above):
mlt_budget_stavg = mlt_budget_fixedh(ds_day_budget)

# Multiply by Delta t for differences:
mlt_budget_stavg = mlt_budget_stavg*(ds_day_budget.average_DT/np.timedelta64(1,'s'))

In [ ]:
# Daily falling average budget:
mlt_budget_falavg = mlt_budget_fixedh(ds_day_budget_falavg)

In [ ]:
# Daily rising average:
mlt_budget_risavg = mlt_budget_stavg - mlt_budget_falavg

In [ ]:
# Define n-1 DataArrays for the months:
n_minus_1 = xr.DataArray(data=[x-1 for x in mlt_budget_risavg.time.dt.day.values],dims=['time'],coords={'time':mlt_budget_stavg.time})

# Monthly rising average:
mlt_budget_risavg_monthly = mlt_budget_risavg.resample(time='1ME').mean() + (mlt_budget_stavg*n_minus_1).resample(time='1ME').mean()

# Monthly standard average:
mlt_budget_stavg_monthly = mlt_budget_stavg.resample(time='1ME').sum()

In [ ]:
# Define function to compute full budget given two months of interest:
def monthly_hat_difference(mlt_budget_risavg_monthly,mlt_budget_stavg_monthly,mlt_monthly,month1_index,month2_index):
    """
    Compute hat difference mlt budget from month 1 to month 2. 

    Inputs:
    - mlt_budget_risavg_monthly: The monthly difference budget, rising average
    - mlt_budget_stavg_monthly: The monthly difference budget, standard average
    - mlt_monthly: The monthly-average mixed layer temperature
    - month1_index: The index of the first month
    - month2_index: The index of the second month
    """

    # List of variables:
    vars = list(mlt_budget_risavg_monthly.data_vars)

    # Interim months:
    monthM_index = np.arange(month1_index+1,month2_index,1)
    
    # Compute mlt difference as "mlt_tendency":
    mlt_budget_hat_diff = (mlt_monthly.isel(time=month2_index) - mlt_monthly.isel(time=month1_index)).rename('mlt_tendency').to_dataset()

    # Compute falling difference as difference between other budgets:
    mlt_budget_falavg_monthly = mlt_budget_stavg_monthly - mlt_budget_risavg_monthly

    # Compute budget terms:
    for var in vars:
        mlt_budget_hat_diff[var] = mlt_budget_risavg_monthly[var].isel(time=month1_index) + mlt_budget_stavg[var].isel(time=monthM_index).sum('time') + mlt_budget_falavg_monthly[var].isel(time=month2_index)

    # Compute entrainment by residual:
    mlt_budget_hat_diff['entrainment'] = -(mlt_budget_hat_diff['fixedh_tendency'] - mlt_budget_hat_diff['mlt_tendency'])

    return(mlt_budget_hat_diff)

## Plot daily-resolution time series plot
This code produces a plot of the mixed layer temperature, mixed layer depth and contributions to the standard and hat-averaged raw budgets at a daily time-scale.

In [ ]:
sreg = [150-360,168-360,-44,-38] # Tasman Sea region from Kajtar et al. 2022

In [ ]:
# climatology:
ds_clim1 = ds_clim.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean']).assign_coords({'time':[np.datetime64('2017-01')+np.timedelta64(x,'M') for x in range(12)]})
ds_clim2 = ds_clim.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean']).assign_coords({'time':[np.datetime64('2018-01')+np.timedelta64(x,'M') for x in range(12)]})
ds_climA = xr.concat([ds_clim1,ds_clim2],dim='time').resample(time='1D').interpolate("linear").sel(time=slice('2017-09-01','2017-12-31'))

#
mlt_budget_stavg_clim1 = mlt_budget_stavg_clim.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean']).assign_coords({'time':[np.datetime64('2017-01')+np.timedelta64(x,'M') for x in range(12)]})
mlt_budget_stavg_clim2 = mlt_budget_stavg_clim.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean']).assign_coords({'time':[np.datetime64('2018-01')+np.timedelta64(x,'M') for x in range(12)]})
mlt_budget_stavg_climA = xr.concat([mlt_budget_stavg_clim1,mlt_budget_stavg_clim2],dim='time').resample(time='1D').interpolate("linear").sel(time=slice('2017-09-01','2017-12-31'))

In [ ]:
fig, axes = plt.subplots(nrows=4,ncols=1,figsize=(12,16),height_ratios=[1.,0.5,1.,1.])

mlt = (ds_day.temp_in_mld/rho0).sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean'])
mlt_snap = (ds_day_snapshot.temp_in_mld/rho0).sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean'])
mlt.plot(ax=axes[0],label='Daily-average mixed layer temperature',linewidth=5.)
mlt_snap.plot(ax=axes[0],label='Snapshot mixed layer temperature',linewidth=2.,linestyle='dashed')
(ds_climA.temp_in_mld/rho0).plot(ax=axes[0],label='1989-2018 climatology',linewidth=2.)

# Plot some averages:
Octavg = mlt.sel(time=slice('2017-10-01','2017-10-31')).mean('time')
Decavg = mlt.sel(time=slice('2017-12-01','2017-12-31')).mean('time')
axes[0].plot([np.datetime64('2017-10-01'),np.datetime64('2017-11-01')],[Octavg.values,Octavg.values],'-',color='C0',linewidth=2.)
axes[0].plot([np.datetime64('2017-12-01'),np.datetime64('2018-01-01')],[Decavg.values,Decavg.values],'-',color='C0',linewidth=2.)
axes[0].plot([np.datetime64('2017-10-16T12:00:00'),np.datetime64('2017-12-16T12:00:00')],[Octavg.values,Decavg.values],'-',color='C0',linewidth=2.,linestyle='dashed',label='Epoch difference (Dec minus Oct)')
axes[0].plot([np.datetime64('2017-10-01'),np.datetime64('2017-12-31')],[mlt_snap.sel(time='2017-10-01',method='nearest'),mlt_snap.sel(time='2017-12-31',method='nearest')],'-',color='C1',linewidth=2.,linestyle='dotted',label='Snapshot difference (24Z 31st Dec - 24Z 1st Oct)')

axes[0].legend()
axes[0].set_title('Tasman Sea mixed layer temperature budget, September-December 2017')
axes[0].set_ylabel('Temperature ($\circ$C)')
axes[0].grid()

ds_day.mld.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean']).plot(ax=axes[1],linewidth=2.,label='Mixed layer depth')
(ds_climA.mld).plot(ax=axes[1],label='1989-2018 climatology',linewidth=2.)
axes[1].set_ylabel('Mixed layer depth (m)')
axes[1].grid()
axes[1].legend()
axes[1].set_ylim([0.,150.])

vars = ['mlt_tendency','entrainment','advection','vert_mixing','surface_flux','sw_pen']
labels = ['Tendency','Entrainment','Advection','Vertical Mixing','Surface fluxes','SW penetration']

budget_stavg = mlt_budget_stavg_daily.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean'])
budget_hatavg = mlt_budget_hatavg_daily.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean'])
budget_stavg_clim = mlt_budget_stavg_climA
unit_conv = 86400

for j, var in enumerate(vars):
    if j == 0:
        (budget_stavg[var]*unit_conv).plot(ax=axes[2],color='C' + str(j),linewidth=2.,label=labels[j] + ' (st. avg.)')
        (budget_hatavg[var]*unit_conv).plot(ax=axes[2],color='C' + str(j),linewidth=2.,linestyle='dashed',label=labels[j] + ' (hat avg.)')
        (budget_stavg_clim[var]*unit_conv).plot(ax=axes[2],color='C' + str(j),linewidth=1.,label=labels[j] + ' (1989-2018 climat.)')
    else:
        (budget_stavg[var]*unit_conv).plot(ax=axes[2],color='C' + str(j),linewidth=2.,label=labels[j])
        (budget_hatavg[var]*unit_conv).plot(ax=axes[2],color='C' + str(j),linewidth=2.,linestyle='dashed')
        (budget_stavg_clim[var]*unit_conv).plot(ax=axes[2],color='C' + str(j),linewidth=1)
axes[2].legend()
axes[2].set_ylabel('Temperature tendency ($\circ$C/day)')
axes[2].grid()

budget_stavg_clim_daily = mlt_budget_stavg_climA.interp(time=budget_stavg.time)
for j, var in enumerate(vars):
    ((budget_stavg[var]-budget_stavg_clim_daily[var])*unit_conv).plot(ax=axes[3],color='C' + str(j),linewidth=2,label=labels[j])
axes[3].legend()
axes[3].set_ylabel('Anomalous temperature \n tendency ($\circ$C/day)')
axes[3].grid()

for ax in axes:
    ax.set_xlabel('')
    ax.set_xlim([mlt.time[0],mlt.time[-1]])
    
plt.savefig('MLT_budget_TasmanSea_OctDec2017_time_series_with_anomalies.png',dpi=250,bbox_inches='tight')

## Spatial plots of time-averaged budgets

This section plots spatial plots of the different contributions to budgets integrated over different time periods

In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=6, figsize=(20,9))
axs = axes.reshape(-1)

vars = ['mlt_tendency','entrainment','advection','vert_mixing','surface_flux','sw_pen']
labels = ['Tendency','Entrainment','Advection','Vertical Mixing','Surface fluxes','SW penetration']
clim = 10

# Standard budget difference terms averaged over 3 month period (equivalent to snapshot difference):
times = slice('2017-10-01','2017-12-31')
stavg_budget = mlt_budget_stavg_daily_monthly.sel(time=times).sum('time')

for j, var in enumerate(vars):
    stavg_budget[var].where(stavg_budget[var]!=0.).plot(ax=axes[0][j],cmap='RdBu_r',vmin=-clim,vmax=clim,extend='both',cbar_kwargs={'label':''}) 
    axes[0][j].set_title('St. Avg ' + labels[j] + ' ($^\circ$C)')
axes[0][0].set_title(axes[0][0].get_title() + '\n (24Z 31st Dec - 24Z 1st Oct snapshot difference)')

# Hat average budget difference terms between 1st and last month (equivalent to time-average difference):
mlt_monthly = (ds_day.temp_in_mld/rho0).resample(time='1M').mean()
hatavg_budget = monthly_hat_difference(mlt_budget_risavg_monthly.sel(time=times),mlt_budget_stavg_monthly.sel(time=times),mlt_monthly.sel(time=times),0,len(mlt_budget_risavg_monthly.time.sel(time=times))-1)

for j, var in enumerate(vars):
    hatavg_budget[var].plot(ax=axes[1][j],cmap='RdBu_r',vmin=-clim,vmax=clim,extend='both',cbar_kwargs={'label':''}) 
    axes[1][j].set_title('Hat. Avg ' + labels[j] + ' ($^\circ$C)')
axes[1][0].set_title(axes[1][0].get_title() + '\n (Dec - Oct average difference)')

# Hat average budget difference terms between 1st and last day (equivalent to time-average difference) as a check:
mlt_daily = (ds_day.temp_in_mld/rho0)
hatavg_budget_daily = monthly_hat_difference(mlt_budget_risavg.sel(time=times),mlt_budget_stavg.sel(time=times),mlt_daily.sel(time=times),0,len(mlt_budget_risavg.time.sel(time=times))-1)

for j, var in enumerate(vars):
    hatavg_budget_daily[var].plot(ax=axes[2][j],cmap='RdBu_r',vmin=-clim,vmax=clim,extend='both',cbar_kwargs={'label':''}) 
    axes[2][j].set_title('Hat. Avg ' + labels[j] + ' ($^\circ$C)')
axes[2][0].set_title(axes[2][0].get_title() + '\n (31st Dec - 1st Oct average difference)')

for ax in axs:
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_facecolor([0.5,0.5,0.5])
    ax.plot([sreg[0],sreg[1],sreg[1],sreg[0],sreg[0]],[sreg[2],sreg[2],sreg[3],sreg[3],sreg[2]],'-k')
    
plt.tight_layout()
plt.savefig('MLT_budget_TasmanSea_OctDec2017_spatial.png',dpi=250,bbox_inches='tight')

# Compare offline and daily budgets (needs updating)

What diagnostics are needed to do the offline binning:
- Closed 3D heat budget
- `dzt` and `mld` time-averages to bin the 3D diagnostics into the mixed layer.
- Snapshots of `temp_in_mld` for tendency calculation.

## Define function to compute offline mixed layer temperature budget

In [ ]:
def compute_mixed_layer_temperature_budget_offline(ds_budget,ds_budget_2d,mld,dzt):
    """
    Compute mixed layer temperature budget

    Inputs:
    ds_budget -> dataset containing time-averaged 3D budget quantities
    ds_budget_2d -> dataset containing time-averaged 2D (surface layer only) budget quantities (these are just added in without summing)
    mld -> dataarray containing time-averaged mixed layer depth
    dzt -> datarray containing time-averaged grid cell thicknesses
    temp_mld_snap -> datarray containing snapshots of temp*rho0*dzt summed over mld
    mld_snap -> dataarray containing snapshots of mld

    Outputs:
    ds_budget_in_mld -> dataset containing all the "internal" (i.e. not including mlt tendency and entrainment) terms in the diagnosed mixed layer temperature budget (2D), in units of degC/second
    """

    # Sum over mixed layer:
    ds_budget_in_mld = ds_budget.isel(st_ocean=0,drop=True).copy(deep=True)
    for var in list(ds_budget_2d.data_vars):
        ds_budget_in_mld[var] = xr.zeros_like(ds_budget['fixedh_tendency']).isel(st_ocean=0,drop=True).copy(deep=True)
    for ti in range(len(ds_budget.time)): # Loop over time
        dzt_ti = dzt.isel(time=ti).load()       
        dzt_ti_bot = dzt_ti.cumsum('st_ocean')  # Depth (from free surface) of bottom of each cell
        ds_budget_ti = ds_budget.isel(time=ti).load()
        mld_ti = mld.isel(time=ti).load()
        for var in list(ds_budget.data_vars):
            ds_budget_in_mld[var][ti,:,:] = ds_budget_ti[var].where(dzt_ti_bot<mld_ti).sum('st_ocean')   # Include all of cells that lie completely in the mixed layer
            for k in range(len(ds_budget_ti.st_ocean)-1):
                ds_budget_in_mld[var][ti,:,:] += xr.where(np.logical_and(dzt_ti_bot[k,:,:]>mld_ti,dzt_ti_bot[k+1,:,:]<mld_ti),((mld_ti-dzt_ti_bot[k,:,:])/dzt_ti[k,:,:])*ds_budget_ti[var][k,:,:],0.) # Include only a fraction of cells that lie partially within the mixed layer
            ds_budget_in_mld[var][ti,:,:] = ds_budget_in_mld[var][ti,:,:]/rho0/Cp/mld_ti # Convert units from Wm-2 to degC/sec

        # Add 2D surface layer variables:
        surf_frac = xr.where(dzt_ti_bot[0,:,:]>mld_ti,mld_ti/dzt_ti[0,:,:],1.)            # In regions where the mixed layer depth is shallower than the thickness of the surface grid cell, take only that fraction from the 2D variables
        for var in list(ds_budget_2d.data_vars):
            ds_budget_in_mld[var][ti,:,:] = (surf_frac*ds_budget_2d[var][ti,:,:]/rho0/Cp/mld_ti).load()

    # Compute residual for check:
    ds_budget_in_mld['residual'] = ds_budget_in_mld['fixedh_tendency'] - ds_budget_in_mld[list(ds_budget_in_mld.data_vars)[1:]].to_array().sum('variable')
        
    return(ds_budget_in_mld)

## Compute daily and monthly offline binned mixed layer temperature budgets (standard averaging):

In [ ]:
# Offline budget terms grouping:
bud_var_grps = {'advection':['temp_advection','temp_submeso','temp_vdiffuse_k33','neutral_diffusion_temp','neutral_gm_temp'],
                'vert_mixing':['temp_vdiffuse_diff_cbt','temp_nonlocal_KPP'],
                'surface_flux':['temp_vdiffuse_sbc','frazil_3d','temp_rivermix'],
                'sw_pen':['sw_heat']}
bud_2d_vars = ['temp_eta_smooth','sfc_hflux_pme']

In [ ]:
# Compute monthly budget, by month:

# Add sfc_hflux_pme from standard monthly diagnostics file:
ds_mon_budget_3d['sfc_hflux_pme'] = ds_mon['sfc_hflux_pme']

mlt_budget_stavg_monthly_offline_uncat = []

# Loop over month:
for ti in tqdm(range(len(ds_mon_budget_3d.time))):
    
    # Group terms:
    ds_mon_budget_3d_reduced = ds_mon_budget_3d.isel(time=slice(ti,ti+1))['temp_tendency'].rename('fixedh_tendency').to_dataset().copy(deep=True)
    for var in bud_var_grps.keys():
        ds_mon_budget_3d_reduced[var] = ds_mon_budget_3d.isel(time=slice(ti,ti+1))[bud_var_grps[var][0]].load()
        if (len(bud_var_grps[var])>1):
            for raw_var in bud_var_grps[var][1:]:
                ds_mon_budget_3d_reduced[var] += ds_mon_budget_3d.isel(time=slice(ti,ti+1))[raw_var].load()

    ds_mon_budget_3d_reduced_2d = ds_mon_budget_3d.isel(time=slice(ti,ti+1))[bud_2d_vars].load()

    # Do monthly computation:
    bud = compute_mixed_layer_temperature_budget_offline(ds_mon_budget_3d_reduced,ds_mon_budget_3d_reduced_2d,ds_mon.mld.isel(time=slice(ti,ti+1)),ds_mon.dzt.isel(time=slice(ti,ti+1)))
    # Add 2D vars to sbc term:
    for var in bud_2d_vars:
        bud['surface_flux'] += bud[var]
        bud = bud.drop_vars([var])

    bud = compute_tendency_entrainment(bud,ds_mon_snapshot.isel(time=slice(ti,ti+2)).temp_in_mld/rho0)

    mlt_budget_stavg_monthly_offline_uncat.append(bud)

mlt_budget_stavg_monthly_offline = xr.concat(mlt_budget_stavg_monthly_offline_uncat,dim='time')

In [ ]:
# Save to file:
mlt_budget_stavg_monthly_offline.to_netcdf(tmp_folder + 'mlt_budget_stavg_monthly_offline_2023.nc')

In [ ]:
%%time
# Compute daily budget, in blocks:
mlt_budget_stavg_daily_offline_uncat = []

bs = 5; tl = len(ds_day_budget_3d.time)
blocks = [range(tl)[x*bs:(x+1)*bs] for x in range(int(np.ceil(tl/bs)))]
blocks_snap = [range(tl+1)[x*bs:(x+1)*bs + 1] for x in range(int(np.ceil(tl/bs)))]

for i in tqdm(np.arange(55,len(blocks)+1)):#range(len(blocks))):
    ds_day_budget_3d_reduced = ds_day_budget_3d.isel(time=blocks[i])['temp_tendency'].rename('fixedh_tendency').to_dataset()
    
    for var in bud_var_grps.keys():
        ds_day_budget_3d_reduced[var] = ds_day_budget_3d.isel(time=blocks[i])[bud_var_grps[var][0]]
        if (len(bud_var_grps[var])>1):
            for raw_var in bud_var_grps[var][1:]:
                ds_day_budget_3d_reduced[var] += ds_day_budget_3d.isel(time=blocks[i])[raw_var]
                
    ds_day_budget_3d_reduced_2d = ds_day_budget_3d.isel(time=blocks[i])[bud_2d_vars]

    ds_day_budget_3d_reduced.load()
    ds_day_budget_3d_reduced_2d.load()
    mld = ds_day.mld.isel(time=blocks[i]).load()
    dzt = ds_day_budget_3d.dzt.isel(time=blocks[i]).load()
    
    bud = compute_mixed_layer_temperature_budget_offline(ds_day_budget_3d_reduced,ds_day_budget_3d_reduced_2d,mld,dzt)
    
    for var in bud_2d_vars:
        bud['surface_flux'] += bud[var]
        bud = bud.drop_vars([var])

    temp_in_mld = (ds_day_snapshot.isel(time=blocks_snap[i]).temp_in_mld/rho0).load()
    bud = compute_tendency_entrainment(bud,temp_in_mld)

    #    mlt_budget_stavg_daily_offline_uncat.append(bud)
    bud.to_netcdf(tmp_folder + 'mlt_budget_stavg_daily_offline_2023_block%03d.nc' % i)

In [ ]:
# Concat and save to a single file:
ds_cat = []

for i in tqdm(range(73)):
    ds = xr.open_dataset(tmp_folder + 'mlt_budget_stavg_daily_offline_2023_block%03d.nc' % i).load()
    ds_cat.append(ds)
ds = xr.concat(ds_cat,dim='time')

ds.to_netcdf(tmp_folder + 'mlt_budget_stavg_daily_offline_2023.nc')

In [ ]:
# Create monthly means from daily files and save back to file:
fname = '/g/data/e14/rmh561/mlt_budget_temporary/mlt_budget_stavg_daily_online_2023.nc'
ds  = xr.open_dataset(fname, chunks={'time':61, 'yt_ocean': 180, 'xt_ocean': 240})
ds_mon = xr.zeros_like(ds.resample(time='1ME').mean()).copy(deep=True)

for var in tqdm(ds.data_vars):
    ds_mon[var] = ds[var].resample(time='1ME').mean().load()
ds_mon = ds_mon.assign_coords({'time':ds.time.resample(time='1ME').mean()})
ds_mon.to_netcdf(fname[:-3] + '_monthly_mean.nc')

In [ ]:
ds_mon.to_netcdf('/g/data/e14/rmh561/mlt_budget_temporary/mlt_budget_stavg_daily_online_2023_monthly_mean.nc')

#### Spatial plots comparing monthly, daily offline and daily:

In [ ]:
# Load from file:
mlt_budget_stavg_monthly_offline = xr.open_dataset('/g/data/e14/rmh561/mlt_budget_temporary/mlt_budget_stavg_monthly_offline_2023.nc').load()
mlt_budget_stavg_daily_offline  = xr.open_dataset('/g/data/e14/rmh561/mlt_budget_temporary/mlt_budget_stavg_daily_offline_2023_monthly_mean.nc').load()
mlt_budget_stavg_daily_online  = xr.open_dataset('/g/data/e14/rmh561/mlt_budget_temporary/mlt_budget_stavg_daily_online_2023_monthly_mean.nc').load()

In [ ]:
regs = {'Warm Pool':[-200, -160, -10, 10],
        'Tasman Sea':[-205, -190, -40, -25],
        'Cold Tongue':[-150,-90,-5,5],
       }

In [ ]:
# Region selection:
fig = plt.figure(figsize=(30,15))
mlt_budget_stavg_monthly_offline['vert_mixing'].mean('time').plot()
for key in regs.keys():
    reg = regs[key]
    plt.plot([reg[0],reg[1],reg[1],reg[0],reg[0]],[reg[2],reg[2],reg[3],reg[3],reg[2]],'-k')
plt.gca().set_facecolor('k')

In [ ]:
# Spatial plots:
fig, axs = plt.subplots(nrows=3, ncols=6, figsize=(15,6),layout='constrained')

vars = ['mlt_tendency','entrainment','advection','vert_mixing','surface_flux','sw_pen']
labels = ['Tendency','Entrainment','Advection','Vertical Mixing','Surface fluxes','SW penetration']
budgets = [mlt_budget_stavg_monthly_offline.mean('time'),
           mlt_budget_stavg_daily_offline.mean('time'),
           mlt_budget_stavg_daily_online.mean('time'),
           #mlt_budget_stavg_daily_online.mean('time') - mlt_budget_stavg_monthly_offline.mean('time'),
           #mlt_budget_stavg_daily_online.mean('time') - mlt_budget_stavg_daily_offline.mean('time')
          ]
budget_labels = ['Monthly Offline','Daily Offline','Online','Online - Monthly Offline','Online - Daily Offline']
unit_conv = 86400*30.5
clims = [1,1,1,2.5,5,5]

for i in range(len(budgets)):
    for j, var in enumerate(vars):
        if i == 0:
            (budgets[i][var]*unit_conv).plot(ax=axs[i][j],cmap='RdBu_r',vmin=-clims[j],vmax=clims[j],extend='both',cbar_kwargs={'label':labels[j] + ' ($^\circ$C/month)','shrink':0.7,'location':'top'})
        else:
            (budgets[i][var]*unit_conv).plot(ax=axs[i][j],cmap='RdBu_r',vmin=-clims[j],vmax=clims[j],extend='both',add_colorbar=False)
        axs[i][j].set_ylabel('')
    axs[i][0].set_ylabel(budget_labels[i])
    
for key in regs.keys():
    reg = regs[key]
    axs[2][0].plot([reg[0],reg[1],reg[1],reg[0],reg[0]],[reg[2],reg[2],reg[3],reg[3],reg[2]],'-k',linewidth=0.5)

for ax in axs.reshape(-1):
    ax.set_ylim([-75,75])
    ax.set_xlabel('')
    ax.set_title('')
    ax.set_facecolor('k')
    ax.set_xticklabels([])
    ax.set_yticklabels([])

plt.savefig('MLT_budget_2023_Online_Offline_Comparison.png',dpi=300)

In [ ]:
# Compute spatial averages:

budgets = {}

def area_average(budget,reg):

    total_area = ds_grid.area_t.sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).mean(['xt_ocean','yt_ocean'])
    budget_av = (budget*ds_grid.area_t).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).mean(['xt_ocean','yt_ocean'])/total_area
    return(budget_av)

for key in tqdm(regs.keys()):
    reg = regs[key]
    budgets[key] = [area_average(mlt_budget_stavg_monthly_offline,reg).load(),
               area_average(mlt_budget_stavg_daily_offline,reg).load(),
               area_average(mlt_budget_stavg_daily_online,reg).load()
              ]

In [ ]:
# Time series across specific regions:
fig, axs = plt.subplots(nrows=len(regs.keys()), ncols=1, figsize=(12,12))

vars = ['mlt_tendency','entrainment','advection','vert_mixing','surface_flux','sw_pen']
labels = ['Tendency','Entrainment','Advection','Vertical Mixing','Surface fluxes','SW penetration']
cols = ['k','r','b','g','m','c']
typs= ['-','--',':']
budget_labels = ['Monthly Offline','Daily Offline','Online']
unit_conv = 86400*30.5

for k, key in enumerate(regs.keys()):
    reg = regs[key]
    for i in range(len(budgets)):
        for j, var in enumerate(vars):
            if j == 0 and k == 0:
                (budgets[key][i][var]*unit_conv).plot(ax=axs[k],color=cols[j],linestyle=typs[i],linewidth=2,label=budget_labels[i])
            elif i == 0 and k == 1:
                (budgets[key][i][var]*unit_conv).plot(ax=axs[k],color=cols[j],linestyle=typs[i],linewidth=2,label=labels[j])
            else:
                (budgets[key][i][var]*unit_conv).plot(ax=axs[k],color=cols[j],linestyle=typs[i],linewidth=2)
    axs[k].set_title(key)
axs[0].legend()
axs[1].legend()
plt.tight_layout()
plt.savefig('MLT_budget_2023_Online_Offline_Comparison_Time_Series.png',dpi=300)

In [ ]:

fig, axes = plt.subplots(nrows=3,ncols=1,figsize=(13, 15))

months = {'Warm Pool':range(12),
        'Tasman Sea':[9,10,11,0,1,2],
        'Cold Tongue':range(12),
       }
month_lab = {'Warm Pool':'Annual',
        'Tasman Sea':'Oct-Dec, Jan-Feb',
        'Cold Tongue':'Annual'}

# Bar plots across specific regions:
for i, key in enumerate(budgets.keys()):
    ax = axes[i]
    
    ds = xr.concat(budgets[key],dim='class').assign_coords({'class':['Monthly Offline','Daily Offline','Online']}).isel(time=months[key]).mean('time')
    
    # Add another variable:
    ds['surface_flux_total'] = ds['surface_flux']+ds['sw_pen']
    ds['vert_total'] = ds['surface_flux']+ds['sw_pen']+ds['vert_mixing']
    
    variables = ['mlt_tendency','entrainment','advection','vert_mixing','surface_flux','sw_pen','residual','surface_flux_total','vert_total']
    classes = ['Monthly Offline','Daily Offline','Online']
    labels = ['Tendency','Entrainment','Advection','Vertical Mixing','Surface fluxes','SW penetration','Residual','Surface fluxes +\nSW penetration','Surface fluxes+ \nSW penetration \n+ Vertical Mixing']
    
    # Convert to DataFrame for easier plotting
    df = ds.to_dataframe()
    
    # Reset index to get 'class' as a column
    df = df.reset_index()
    
    # Parameters
    num_vars = len(variables)
    num_classes = len(classes)
    bar_width = 0.25
    x = np.arange(num_vars)  # One x position per variable
    
    unit_conv = 86400*30.5
    # Create figure
    
    # Plot each class as a separate bar group
    for i, cls in enumerate(classes):
        # Get values for this class across all variables
        values = [df[df['class'] == cls][var].values[0]*unit_conv for var in variables]
        
        # Offset x positions for each class
        ax.bar(x + i * bar_width, values, width=bar_width, label=cls)
    
    # Formatting
    ax.set_xticks(x + bar_width)
    ax.set_xticklabels(labels)
    ax.set_ylabel("$^\circ$C/month")
    ax.set_title(key + ' Mixed Layer Temperature Budget ' +month_lab[key] + ' 2023')
    ax.legend(title="Budget")
    ax.grid()
    plt.tight_layout()
plt.savefig('MLT_budget_Online_Offline_Comparison_Bar.png',dpi=250)

# Testing and checks

## Test that Eulerian budget (no MLD binning) closes:

In [ ]:
# Choose time, region, depth:
time = 0; reg_slice= [-300, 300,-90,90]; st_ocean = 40;

# Load budget:
ds_day_budget_slice = xr.open_dataset(base2 + 'ocean_budget_daily.nc').sel(xt_ocean=slice(reg_slice[0],reg_slice[1]),yt_ocean=slice(reg_slice[2],reg_slice[3])).isel(time=time).isel(st_ocean=st_ocean)

# List the terms:
bud_vars = ['temp_advection','temp_submeso','neutral_diffusion_temp','neutral_gm_temp','temp_vdiffuse_k33',
            'temp_nonlocal_KPP','temp_vdiffuse_diff_cbt',
            'temp_rivermix','temp_vdiffuse_sbc', 'frazil_3d', 
            'sw_heat']

# Add the 2D terms if we're looking at the surface layer:
if st_ocean == 0:
    bud_vars = bud_vars + ['temp_eta_smooth','sfc_hflux_pme']

# Compute the residual:
ds_day_budget_slice['residual'] = ds_day_budget_slice.temp_tendency.load().copy(deep=True)
for var in bud_vars:
    ds_day_budget_slice['residual'] -= ds_day_budget_slice[var].load()

# Add tendency and residual to the terms list:
bud_vars = ['temp_tendency','residual'] + bud_vars

In [ ]:
# Plot every term and print out the maximum of the absolute value of every term to confirm closure:
fig, axes = plt.subplots(nrows=4,ncols=4,figsize=(25,20))
axs = axes.reshape(-1)
print('Spatial maximums of terms (Wm-2):')
for i, var in enumerate(bud_vars):
    ds_day_budget_slice[var].plot(ax=axs[i],vmin=-10.,vmax=10.,cmap='RdBu_r')
    axs[i].set_title(var)
    print('%10.5f, ' % (abs(ds_day_budget_slice[var]).max().values) + ' ' + var)


## Test that MLD binned budgets (standard and falling averages) close:

In [ ]:
# Choose time and region:
time = 0; reg_slice = [-300, 300,-90,90]#reg = [-100, 20, 0, 60]

# List the terms:
bud_vars = ['temp_tendency_in_mld',
            'sfc_hflux_pme_in_mld','temp_eta_smooth_in_mld','temp_advection_in_mld','temp_submeso_in_mld','neutral_diffusion_in_mld_temp','neutral_gm_in_mld_temp','temp_vdiffuse_k33_in_mld',
            'temp_nonlocal_KPP_in_mld','temp_vdiffuse_diff_cbt_in_mld',
            'temp_rivermix_in_mld','temp_vdiffuse_sbc_in_mld', 'frazil_3d_in_mld', 
            'sw_heat_in_mld']

# Load budget:
# Standard average:
# ds_day_budget_slice = xr.open_dataset(base2 + 'ocean_budget_daily.nc')[bud_vars].sel(xt_ocean=slice(reg_slice[0],reg_slice[1]),yt_ocean=slice(reg_slice[2],reg_slice[3])).isel(time=time)

# Falling average:
ds_day_budget_slice = xr.open_dataset(base2 + 'ocean_budget_daily_risavg.nc')[bud_vars].sel(xt_ocean=slice(reg_slice[0],reg_slice[1]),yt_ocean=slice(reg_slice[2],reg_slice[3])).isel(time=time)/86400.

# Note: For the daily-binned budget the 2D terms (temp_eta_smooth and sfc_hflux_pme) are explicitly binned because the mixed layer depth can be less than the depth of the top grid cell (in which case we don't want to take the whole budget term).

# Compute residual:
ds_day_budget_slice['residual_in_mld'] = ds_day_budget_slice.temp_tendency_in_mld.load().copy(deep=True)
for var in bud_vars[1:]:
    ds_day_budget_slice['residual_in_mld'] -= ds_day_budget_slice[var].load()

# Add tendency and residual to budget list:
bud_vars = ['residual_in_mld'] + bud_vars

In [ ]:
# Plot every term and print out the maximum of the absolute value of every term to confirm closure:
fig, axes = plt.subplots(nrows=4,ncols=4,figsize=(25,20))
axs = axes.reshape(-1)
print('Spatial maximums of terms (Wm-3):')
for i, var in enumerate(bud_vars):
    ds_day_budget_slice[var].plot(ax=axs[i],vmin=-.1,vmax=.1,cmap='RdBu_r')
    axs[i].set_title(var)
    print('%10.5f, ' % (abs(ds_day_budget_slice[var]).max().values) + ' ' + var)


## Bug in temp_mld diagnostic demonstration (see https://github.com/mom-ocean/MOM5/issues/397):

In [ ]:
fig,axes = plt.subplots(nrows=3,ncols=2,figsize=(15,15))
(ds_month.mld).isel(time=0).plot(vmin=0.,vmax=100.,ax=axes[0][0],cmap=cm.cm.amp)
axes[0][0].set_title('MLD [m]')
(ds_month.temp_mld/ds_day.mld/rho0).isel(time=0).plot(vmin=0.,vmax=30,ax=axes[1][0],cmap=cm.cm.thermal)
axes[1][0].set_title('MLT from temp_mld [degC]')
(ds_month.temp_avg_mld/rho0).isel(time=0).plot(vmin=0.,vmax=30.,ax=axes[0][1],cmap=cm.cm.thermal)
axes[0][1].set_title('MLT from temp_avg_mld [degC]')
(ds_month.temp_in_mld/rho0).isel(time=0).plot(vmin=0.,vmax=30.,ax=axes[1][1],cmap=cm.cm.thermal)
axes[1][1].set_title('MLT from temp_in_mld [degC]')
(ds_month.temp-273.15).isel(time=0,st_ocean=0).plot(vmin=0.,vmax=30.,ax=axes[2][0],cmap=cm.cm.thermal)
axes[2][0].set_title('SST [degC]')
(ds_month.temp.isel(st_ocean=0)-273.15 - ds_month.temp_in_mld/rho0).isel(time=0).plot(vmin=-0.05,vmax=0.05,ax=axes[2][1],cmap='RdBu_r')
axes[2][1].set_title('SST - MLT from temp_in_mld [degC]')
plt.savefig('temp_mld_diagnostic_checks_global.png',dpi=250)

## Hat-averaging, single-output tendencies check

In [ ]:
# 3D diagnostics:
HC_tend_stavg = ds_day_budget.temp_tendency.isel(st_ocean=0)
HC_tend_falavg = ds_day_budget_falavg.temp_tendency.isel(st_ocean=0)/(ds_day_budget_falavg.average_DT/np.timedelta64(1,'s')) # Division by Dt in seconds required because falling average is an integral not a sum.
HC_tend_risavg = HC_tend_stavg - HC_tend_falavg # Rising average = standard average - falling average

# Time-averaged HC and tendency:
HC = ds_day_budget.temp_rhodzt.isel(st_ocean=0)*Cp
HC_tend_from_HC = HC.diff('time')/86400.
time_cen = [HC.time.isel(time=slice(x,x+2)).mean('time').values for x in range(len(HC.time))][:-1]
HC_tend_from_HC = HC_tend_from_HC.assign_coords({'time':time_cen})

# Time-snapshot HC and tendency:
HC_snap = (ds_day_snapshot.temp.isel(st_ocean=0)-273.15)*rho0*ds_day_snapshot.dzt.isel(st_ocean=0)*Cp
HC_snap_tend_from_HC = HC_snap.diff('time')/86400.
HC_snap_tend_from_HC = HC_snap_tend_from_HC.assign_coords({'time':HC.time.values})

# Time-averaged tendency from hat average:
HC_tend_hatavg = xr.zeros_like(HC_tend_hatavg_from_HC)
HC_tend_hatavg.data = HC_tend_risavg.isel(time=slice(0,-1)).values + HC_tend_falavg.isel(time=slice(1,None)).values

In [ ]:
# Plot at a point
xt = 20
yt = 1
times = slice(0,14)

fig, axes = plt.subplots(nrows=2,ncols=2,figsize=(14,10))

HC_snap.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[0][0],label='Snapshot HC')
HC.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[0][0],label='Daily-averaged HC')
HC_snap.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[0][1],label='Snapshot HC')
HC.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[0][1],label='Daily-averaged HC')

HC_snap_tend_from_HC.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[1][0],label='d HC_snap/dt',linewidth=3.)
HC_tend_from_HC.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[1][1],label='d HC/ dt',linewidth=3.)
HC_tend_hatavg.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[1][1],label='Tendency, hat average')
HC_tend_stavg.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[1][0],label='Tendency, standard average')
HC_tend_risavg.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[1][0],label='Tendency, rising average')
HC_tend_falavg.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[1][0],label='Tendency, falling average')
for ax in axes.reshape(-1):
    ax.grid()
    ax.set_xlim(axes[0][0].get_xlim())
    ax.legend()

In [ ]:
fig, axes = plt.subplots(nrows=2,ncols=3,figsize=(20,10))
index = 1

HC_tend_stavg.isel(time=index).plot(ax=axes[0][0],vmin=-30.,vmax=30.,cmap='RdBu_r')
axes[0][0].set_title('Tendency standard average (Wm-2)')
HC_tend_risavg.isel(time=index).plot(ax=axes[0][1],vmin=-30.,vmax=30.,cmap='RdBu_r')
axes[0][1].set_title('Tendency rising average (Wm-2)')
HC_tend_falavg.isel(time=index).plot(ax=axes[0][2],vmin=-30.,vmax=30.,cmap='RdBu_r')
axes[0][2].set_title('Tendency falling average (Wm-2)')
HC_tend_from_HC.isel(time=0).plot(ax=axes[1][0],vmin=-30.,vmax=30.,cmap='RdBu_r')
axes[1][0].set_title('Tendency from daily-average HC (Wm-2)')
HC_tend_hatavg.isel(time=0).plot(ax=axes[1][1],vmin=-30.,vmax=30.,cmap='RdBu_r')
axes[1][1].set_title('Tendency hat average (Wm-2)')
(HC_tend_from_HC - HC_tend_hatavg).isel(time=0).plot(ax=axes[1][2])
axes[1][2].set_title('Difference')

## Hat-averaging, combined-outputs epoch difference check

In [ ]:
# single-output standard, rising and falling averages:
HC_tend_stavg = ds_day_budget.temp_tendency.isel(st_ocean=0)*(ds_day_budget.average_DT/np.timedelta64(1,'s')) # Short period standard difference (hence x DT)
HC_tend_falavg = ds_day_budget_falavg.temp_tendency.isel(st_ocean=0)                                        # Short period falling average difference (already x DT in code)
HC_tend_risavg = HC_tend_stavg - HC_tend_falavg                                                             # Short period rising average difference (standard - falling)

# Time-averaged heat content and its single-output tendency:
HC = ds_day_budget.temp_rhodzt.isel(st_ocean=0)*Cp
HC_tend_from_HC = HC.diff('time')/86400.
time_cen = [HC.time.isel(time=slice(x,x+2)).mean('time').values for x in range(len(HC.time))][:-1]
HC_tend_from_HC = HC_tend_from_HC.assign_coords({'time':time_cen})

In [ ]:
# Define epoch periods:
epoch1 = list(range(10))
epoch2 = [len(HC.time) - 5 + x for x in range(5)]
epochM = np.arange(epoch1[-1]+1,epoch2[0],1)

n_minus_1_epoch1 = xr.DataArray(data=range(len(epoch1)),dims=['time'],coords={'time':HC_tend_stavg.time.isel(time=epoch1)}) # (n-1) for epoch1 as a DataArray
n_minus_1_epoch2 = xr.DataArray(data=range(len(epoch2)),dims=['time'],coords={'time':HC_tend_stavg.time.isel(time=epoch2)}) # (n-1) for epoch2 as a DataArray

# Compute epoch HC change from HC:
HC_change = HC.isel(time=epoch2).mean('time') - HC.isel(time=epoch1).mean('time') # All days have same length, so no weighted mean is needed

# Compute epoch HC change from hat average tendencies:
long_ris_epoch1 = HC_tend_risavg.isel(time=epoch1).mean('time') + (HC_tend_stavg.isel(time=epoch1)*n_minus_1_epoch1).mean('time')
long_sta_epoch1 = HC_tend_stavg.isel(time=epoch1).sum('time')
long_fal_epoch1 = long_sta_epoch1 - long_ris_epoch1

long_sta_epochM = HC_tend_stavg.isel(time=epochM).sum('time')

long_ris_epoch2 = HC_tend_risavg.isel(time=epoch2).mean('time') + (HC_tend_stavg.isel(time=epoch2)*n_minus_1_epoch2).mean('time')
long_sta_epoch2 = HC_tend_stavg.isel(time=epoch2).sum('time')
long_fal_epoch2 = long_sta_epoch2 - long_ris_epoch2

HC_change_from_hatavg = long_ris_epoch1 + long_sta_epochM + long_fal_epoch2

In [ ]:
# Plot at a point
xt = 20
yt = 1
times = slice(0,len(HC.time))

fig = plt.figure(figsize=(9,5))
axes = [plt.gca()]

HC.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[0],label='Daily-averaged HC')
axes[0].plot(HC.time.isel(time=epoch1),HC.isel(xt_ocean=xt,yt_ocean=yt,time=epoch1).mean('time').values*xr.ones_like(HC.isel(xt_ocean=xt,yt_ocean=yt,time=epoch1)),linewidth=3.,color='C0')
axes[0].text(HC.time.isel(time=epoch1[0]),HC.isel(xt_ocean=xt,yt_ocean=yt,time=epoch1).mean('time').values,'Epoch 1')
axes[0].plot(HC.time.isel(time=epoch2),HC.isel(xt_ocean=xt,yt_ocean=yt,time=epoch2).mean('time').values*xr.ones_like(HC.isel(xt_ocean=xt,yt_ocean=yt,time=epoch2)),linewidth=3.,color='C0')
axes[0].text(HC.time.isel(time=epoch2[0]),HC.isel(xt_ocean=xt,yt_ocean=yt,time=epoch2).mean('time').values,'Epoch 2')

axes[0].text(np.datetime64('2023-06-01'),0.6e7,'%5.0f = Epoch 1 rising' % long_ris_epoch1.isel(xt_ocean=xt,yt_ocean=yt).values)
axes[0].text(np.datetime64('2023-06-01'),0.65e7,'%5.0f = Epoch M standard' % long_sta_epochM.isel(xt_ocean=xt,yt_ocean=yt).values)
axes[0].text(np.datetime64('2023-06-01'),0.7e7,'%5.0f = Epoch 2 falling' % long_fal_epoch2.isel(xt_ocean=xt,yt_ocean=yt).values)
axes[0].text(np.datetime64('2023-06-01'),0.75e7,'%5.0f = HC change from hat' % HC_change_from_hatavg.isel(xt_ocean=xt,yt_ocean=yt).values)
axes[0].text(np.datetime64('2023-06-01'),0.8e7,'%5.0f = HC change' % HC_change.isel(xt_ocean=xt,yt_ocean=yt).values)

for ax in axes:
    ax.grid()
    ax.set_xlim(axes[0].get_xlim())
    ax.legend()

In [ ]:
# Plot spatial slice:
fig, axes = plt.subplots(nrows=1,ncols=3,figsize=(20,6))

HC_change.plot(ax=axes[0],vmin=-1.e7,vmax=1.e7,cmap='RdBu_r')
axes[0].set_title('HC change (Jm-2)')
HC_change_from_hatavg.plot(ax=axes[1],vmin=-1.e7,vmax=1.e7,cmap='RdBu_r')
axes[1].set_title('HC change from hat avg (Jm-2)')
(HC_change_from_hatavg-HC_change).plot(ax=axes[2],vmin=-1.e2,vmax=1.e2,cmap='RdBu_r')
axes[2].set_title('Difference')

##### Some old code from Chris that I probably don't need

In [ ]:
# Use a hack of Chris's code to do the hatavg from tendency calculation:

# First, get times (in seconds):
time_stamp_file = base2 + 'time_stamp.out'

def init_run_time(time_stamp_file):
    with open(time_stamp_file, "r") as tstamp:
        tls = tstamp.readline().split()
        tls = [int(_) for _ in tls[:-1]]
        t0_date = datetime.datetime(tls[0], tls[1], tls[2], hour=tls[3], minute=tls[4], second=tls[5])
    time_units = datetime.datetime(1,1,1,0,0,0)
    return float((t0_date - time_units).total_seconds())

init_time = init_run_time(time_stamp_file)
init_times = init_time + (ds_day_budget.average_DT/np.timedelta64(1,'s')).cumsum() - 86400.
final_times = init_time + (ds_day_budget.average_DT/np.timedelta64(1,'s')).cumsum()

# Second,
# def compute_long_ris_avg_fwd(oheat_diag, ora_diag, monthly_avs, final_month_times):
#     """oheat_diag: time-average diagnostic
#     ora_diag: rising average diagnostic
#     monthly_avs: length of month
#     final month times: array of the last timestep of each averaging period (including timestep before run)"""
#     temp_tend_stnd_av = (oheat_diag*monthly_avs).sum('time')/(365*60*60*24) # 1/N * sum(std_av*month)
#     t_weight_av = ora_diag - oheat_diag*final_month_times 
#     t_weight_av_total = (t_weight_av*monthly_avs/(365*60*60*24)).sum('time') + final_month_times[-1]*temp_tend_stnd_av 
#     return t_weight_av_total

risavg_fwd_t0 = HC_tend_risavg.isel(time=0) - HC_tend_stavg.isel(time=0)*final_time.isel(time=0).values + final_time.isel(time=0).values*HC_tend_stavg.isel(time=0)
# def compute_long_ris_avg_bwd(oheat_diag, ora_diag, monthly_avs, initial_month_times):
#     """oheat_diag: time-average diagnostic
#     ora_diag: rising average diagnostic
#     monthly_avs: length of month
#     initial month times: array of the first timestep of each averaging period (including 1st timestep)"""
#     temp_tend_stnd_av = np.sum(oheat_diag*monthly_avs)/(365*60*60*24) # 1/N * sum(std_av*month)
#     t_weight_av = ora_diag + oheat_diag*initial_month_times # ris_avg + std_av*init_times
#     # m/N*(ris_avg + std_av*init_times) - t1*total_av
#     t_weight_av_total = np.sum(t_weight_av*monthly_avs/(365*60*60*24)) - initial_month_times[0]*temp_tend_stnd_av 
#     return t_weight_av_total

risavg_bwd_t1 = (HC_tend_risavg.isel(time=slice(0,2)) + HC_tend_stavg.isel(time=slice(0,2))*init_times.isel(time=slice(0,2))).mean('time') - init_times[-1]*HC_tend_stavg.isel(time=slice(0,2)).mean('time')


In [ ]:
# Chris's code:
# compute long period rising average
def compute_long_ris_avg_fwd(oheat_diag, ora_diag, monthly_avs, final_month_times):
    """oheat_diag: time-average diagnostic
    ora_diag: rising average diagnostic
    monthly_avs: length of month
    final month times: array of the last timestep of each averaging period (including timestep before run)"""
    temp_tend_stnd_av = np.sum(oheat_diag*monthly_avs)/(365*60*60*24) # 1/N * sum(std_av*month)
    t_weight_av = ora_diag - oheat_diag*final_month_times 
    t_weight_av_total = np.sum(t_weight_av*monthly_avs/(365*60*60*24)) + final_month_times[-1]*temp_tend_stnd_av 
    return t_weight_av_total

def compute_long_ris_avg_bwd(oheat_diag, ora_diag, monthly_avs, initial_month_times):
    """oheat_diag: time-average diagnostic
    ora_diag: rising average diagnostic
    monthly_avs: length of month
    initial month times: array of the first timestep of each averaging period (including 1st timestep)"""
    temp_tend_stnd_av = np.sum(oheat_diag*monthly_avs)/(365*60*60*24) # 1/N * sum(std_av*month)
    t_weight_av = ora_diag + oheat_diag*initial_month_times # ris_avg + std_av*init_times
    # m/N*(ris_avg + std_av*init_times) - t1*total_av
    t_weight_av_total = np.sum(t_weight_av*monthly_avs/(365*60*60*24)) - initial_month_times[0]*temp_tend_stnd_av 
    return t_weight_av_total

def init_run_time(time_stamp_file):
    with open(time_stamp_file, "r") as tstamp:
        tls = tstamp.readline().split()
        tls = [int(_) for _ in tls[:-1]]
        t0_date = datetime.datetime(tls[0], tls[1], tls[2], hour=tls[3], minute=tls[4], second=tls[5])
    time_units = datetime.datetime(1,1,1,0,0,0)
    return float((t0_date - time_units).total_seconds())